In [6]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from qutip_qip.operations import (berkeley, cnot, cphase, csign, fredkin,
                                  gate_sequence_product, globalphase, iswap,
                                  molmer_sorensen, phasegate, qrot, rx, ry, rz,
                                  snot, sqrtiswap, sqrtnot, sqrtswap, swap,
                                  swapalpha, toffoli)
from qutip import gates

In [7]:
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
T = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]])
cnot_gate = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0],
                      [0, 0, 0, 1],
                      [0, 0, 1, 0]])
cz_gate = np.array([[1, 0, 0, 0],
                    [0, 1, 0, 0],
                    [0, 0, 1, 0],
                    [0, 0, 0, -1]])

In [8]:
from numpy import size


def string_to_state(s):
    """
    Convert string to state, takes 0,1,+,-
    """
    states = []
    for i in s:
        if i == "0":
            states.append(np.array([[1], [0]]))
        elif i == "1":
            states.append(np.array([[0], [1]]))
        elif i == "+":
            states.append(np.array([[1], [1]]) / np.sqrt(2))
        elif i == "-":
            states.append(np.array([[1], [-1]]) / np.sqrt(2))
    for state in states:
        if 'result' in locals():
            result = np.kron(result, state)
        else:
            result = state
    return result
def Hgate(n):
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    step = H
    for i in range(n-1):
        step = np.kron(step, H)
    return step

def Rx(theta, size):
    gate = np.array([[np.cos(theta/2), -1j*np.sin(theta/2)], [-1j*np.sin(theta/2), np.cos(theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def Ry(theta, size):
    gate = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def Rz(theta, size):
    gate = np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])
    step = gate
    for i in range(size-1):
        step = np.kron(step, gate)
    return step

def gate_on_target(gate, target, size):
    I = np.eye(2)
    factors = [gate if i == target else I for i in range(size)]
    result = factors[0]
    for i in range(1, size):
        result = np.kron(result, factors[i])
    return result

def H_target(state, target, size):
    H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
    I = np.eye(2)
    factors = [H if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def X_target(state, target, size):
    X = np.array([[0, 1], [1, 0]])
    I = np.eye(2)
    factors = [X if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def Z_target(state, target, size):
    Z = np.array([[1, 0], [0, -1]])
    I = np.eye(2)
    factors = [Z if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def Rx_target(state, theta, target, size):
    gate = np.array([[np.cos(theta/2), -1j*np.sin(theta/2)], [-1j*np.sin(theta/2), np.cos(theta/2)]])
    I = np.eye(2)
    factors = [gate if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def Ry_target(state, theta, target, size):
    gate = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
    I = np.eye(2)
    factors = [gate if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def Rz_target(state, theta, target, size):
    gate = np.array([[np.exp(-1j*theta/2), 0], [0, np.exp(1j*theta/2)]])
    I = np.eye(2)
    factors = [gate if i == target else I for i in range(size)]
    gate = factors[0]
    for i in range(1, size):
        gate = np.kron(gate, factors[i])
    return gate @ state

def CNOT(control, target, size):
    """
    (|0><0|)_c ⊗ I_rest + (|1><1|)_c ⊗ X_t ⊗ I_rest
    """
    P0 = np.array([[1, 0], [0, 0]]) #checks if qbit is 0, if yes, keep
    P1 = np.array([[0, 0], [0, 1]]) #checks if qbit is 1, if yes, apply X to target
    I  = np.eye(2)
    X  = np.array([[0, 1], [1, 0]])
    # Term 1: Control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(size)]
    
    # Term 2: Control is |1> -> apply X to target
    term1 = [X if i == target else (P1 if i == control else I) for i in range(size)]

    #compute tensor
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    return op0 + op1

def CZ(control, target, size):
    """
    (|0><0|)_c ⊗ I_rest + (|1><1|)_c ⊗ Z_t ⊗ I_rest
    """
    P0 = np.array([[1, 0], [0, 0]]) #checks if qbit is 0, if yes, keep
    P1 = np.array([[0, 0], [0, 1]]) #checks if qbit is 1, if yes, apply Z to target
    I  = np.eye(2)
    Z  = np.array([[1, 0], [0, -1]])
    #control is |0> -> apply Identity to target
    term0 = [P0 if i == control else I for i in range(size)]
    # Control is |1> -> apply Z to target
    term1 = [Z if i == target else (P1 if i == control else I) for i in range(size)]
    #compute tensor
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    return op0 + op1

def measurement(state, qbit):
    size = int(np.log2(state.shape[0]))
    #extract the state of the qubit to be measured
    P0 = np.array([[1, 0], [0, 0]]) # |0><0|
    P1 = np.array([[0, 0], [0, 1]]) # |1><1|
    I  = np.eye(2)
    term0= [P0 if i == qbit else I for i in range(size)]
    term1= [P1 if i == qbit else I for i in range(size)]
    #make to tensor 
    op0 = term0[0]
    op1 = term1[0]
    for i in range(1, size):
        op0 = np.kron(op0, term0[i])
        op1 = np.kron(op1, term1[i])
    #probabilities
    p0 = np.real(np.conj(state.T) @ op0 @ state)[0, 0] # prob of measuring |0>
    p1 = np.real(np.conj(state.T) @ op1 @ state)[0, 0] # prob of measuring |1>
    #randomly choose outcome
    outcome = np.random.choice([0, 1], p=[p0, p1])
    #collapse state
    if outcome == 0:
        new_state = op0 @ state / np.sqrt(p0)
    else:
        new_state = op1 @ state / np.sqrt(p1)
    return outcome, new_state, [p0, p1]

def density_matrix(state):
    rho = state @ np.conj(state.T)
    return rho

def trace_out(rho, qubit, size):
    """
    Trace out a qubit from a density matrix. CHATTET
    """
    # Reshape the density matrix to separate the qubit to be traced out
    rho_reshaped = rho.reshape([2] * (2 * size))
    # Move the qubit to be traced out to the last position
    axes = list(range(2 * size))
    axes.remove(qubit)
    axes.append(qubit)
    rho_permuted = np.transpose(rho_reshaped, axes)
    # Reshape to combine the remaining qubits
    new_shape = [2 ** (size - 1), 2 ** (size - 1), 2, 2]
    rho_permuted = rho_permuted.reshape(new_shape)
    # Trace out the last two dimensions (the qubit)
    rho_traced_out = np.trace(rho_permuted, axis1=2, axis2=3)
    return rho_traced_out




In [9]:

class QCircuit:
    def __init__(self, num_qubits):
        self.num_qubits = num_qubits
        self.state = np.zeros((2**num_qubits, 1), dtype=complex)
        self.state[0, 0] = 1 

    def apply_gate(self, gate, target):
        # Accept either a callable gate function (like H_target)
        # or a full gate matrix (ndarray).
        if callable(gate):
            self.state = gate(self.state, target, self.num_qubits)
        else:
            # assume `gate` is a full operator matrix acting on the whole system
            self.state = gate @ self.state

    def measure(self, qubit):
        outcome, self.state, probabilities = measurement(self.state, qubit)   
        return outcome, probabilities

    def apply_cnot(self, control, target):
        self.state = CNOT(control, target, self.num_qubits) @ self.state

    def bell_state(self, qubit1, qubit2):
        self.apply_gate(H_target, qubit1)
        self.apply_cnot(qubit1, qubit2)

    def rotation(self, theta, axis, target):
        if axis == 'x':
            self.state = Rx_target(self.state, theta, target, self.num_qubits)
        elif axis == 'y':
            self.state = Ry_target(self.state, theta, target, self.num_qubits)
        elif axis == 'z':
            self.state = Rz_target(self.state, theta, target, self.num_qubits)


In [10]:
#State teleportation circuit
q = QCircuit(3)
#Bell state of 1 and 2
# q.apply_gate(X_target, 0)  # |1> in qbit 0 instead 
print("Initial state of qubits 0, 1, and 2:", q.state)
q.bell_state(1, 2)
print("Initial state of qubits 1 and 2 (Bell state):", q.state)
q.apply_cnot(0, 1)
q.apply_gate(H_target, 0)
outcome0, prob0 = q.measure(0)
outcome1, prob1 = q.measure(1)
if outcome1 == 1: # if 1 then apply X
    q.apply_gate(X_target, 2)
if outcome0 == 1: # if 1 then apply Z
    q.apply_gate(Z_target, 2)
print("Final state of qubit 2 after teleportation:", q.state)
outcome2, prob2 = q.measure(2)
print("Outcome of measurement on qubit 2:", outcome2)
print("Probabilities of measurement on qubit 2:", prob2)


Initial state of qubits 0, 1, and 2: [[1.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]]
Initial state of qubits 1 and 2 (Bell state): [[0.70710678+0.j]
 [0.        +0.j]
 [0.        +0.j]
 [0.70710678+0.j]
 [0.        +0.j]
 [0.        +0.j]
 [0.        +0.j]
 [0.        +0.j]]
Final state of qubit 2 after teleportation: [[1.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]
 [0.+0.j]]
Outcome of measurement on qubit 2: 0
Probabilities of measurement on qubit 2: [np.float64(1.0), np.float64(0.0)]


In [11]:
#Ladder ansatz circuit
size = 3
q = QCircuit(size)
theta = np.pi / 4
for i in range(size):
    q.rotation(theta, 'y', i)
q.apply_cnot(0, 1)
q.apply_cnot(1, 2)
for i in range(3):
    q.rotation(theta, 'y', i)
q.apply_cnot(0, 1)
q.apply_cnot(1, 2)
print(q.state)


[[0.47855339+0.j]
 [0.40533009+0.j]
 [0.375     +0.j]
 [0.1982233 +0.j]
 [0.125     +0.j]
 [0.5517767 +0.j]
 [0.125     +0.j]
 [0.3017767 +0.j]]


In [12]:
from qutip_qip.circuit import QubitCircuit
from qutip_qip.operations import Gate
qc = QubitCircuit(N=3, reverse_states=False)
theta = np.pi / 4
cnot_gate1 = Gate("CNOT", targets=[1], controls=[0])
cnot_gate2 = Gate("CNOT", targets=[2], controls=[1])
ry_gate1 = Gate("RY", targets=[0], arg_value=theta)
ry_gate2 = Gate("RY", targets=[1], arg_value=theta)
ry_gate3 = Gate("RY", targets=[2], arg_value=theta)
qc.add_gate(ry_gate1)
qc.add_gate(ry_gate2)
qc.add_gate(ry_gate3)
qc.add_gate(cnot_gate1)
qc.add_gate(cnot_gate2)
qc.add_gate(ry_gate1)
qc.add_gate(ry_gate2)
qc.add_gate(ry_gate3)
qc.add_gate(cnot_gate1)
qc.add_gate(cnot_gate2)
# qc.draw()
zero_state = qt.tensor(qt.basis(2, 0), qt.basis(2, 0), qt.basis(2, 0))
result_qutip = qc.run(zero_state)
print(result_qutip)


Quantum object: dims=[[2, 2, 2], [1]], shape=(8, 1), type='ket', dtype=Dense
Qobj data =
[[0.47855339]
 [0.40533009]
 [0.375     ]
 [0.1982233 ]
 [0.125     ]
 [0.5517767 ]
 [0.125     ]
 [0.3017767 ]]


In [13]:
import qiskit as qk
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector
qck = qk.QuantumCircuit(3)
theta = Parameter('theta')
# theta = np.pi / 4
qck.ry(theta, 0)
qck.ry(theta, 1)
qck.ry(theta, 2)
qck.cx(0,1)
qck.cx(1,2)
qck.ry(theta, 0)
qck.ry(theta, 1)
qck.ry(theta, 2)
qck.cx(0,1)
qck.cx(1,2)
qck.draw()
qck.parameters
bc = qck.assign_parameters({theta: np.pi / 4})
final_state_qiskit = Statevector(bc)
print(final_state_qiskit)
# qck.draw()



Statevector([0.47855339+0.j, 0.125     +0.j, 0.375     +0.j,
             0.125     +0.j, 0.40533009+0.j, 0.5517767 +0.j,
             0.1982233 +0.j, 0.3017767 +0.j],
            dims=(2, 2, 2))


In [14]:
print(q.state )
print(result_qutip.full())
print(final_state_qiskit)


[[0.47855339+0.j]
 [0.40533009+0.j]
 [0.375     +0.j]
 [0.1982233 +0.j]
 [0.125     +0.j]
 [0.5517767 +0.j]
 [0.125     +0.j]
 [0.3017767 +0.j]]
[[0.47855339+0.j]
 [0.40533009+0.j]
 [0.375     +0.j]
 [0.1982233 +0.j]
 [0.125     +0.j]
 [0.5517767 +0.j]
 [0.125     +0.j]
 [0.3017767 +0.j]]
Statevector([0.47855339+0.j, 0.125     +0.j, 0.375     +0.j,
             0.125     +0.j, 0.40533009+0.j, 0.5517767 +0.j,
             0.1982233 +0.j, 0.3017767 +0.j],
            dims=(2, 2, 2))
